In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAW_FILE = Path("data/raw/final_merged_data.csv")

print("Working directory should be repo root.")
print("Raw file exists:", RAW_FILE.exists())


Working directory should be repo root.
Raw file exists: True


In [2]:
raw_df = pd.read_csv(RAW_FILE)

print("Raw dataframe shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())

relevant_events = [
    "Stimuli Presentation",
    "Memory Quiz Response",
    "Stimuli Placed",
    "Memory Placement confidence",
    "Memory Quiz2 Response",
    "Memory Quiz2 confidence",
]

df_events = raw_df[
    raw_df["V3"].astype(str).str.contains("|".join(relevant_events), na=False)
].copy()

print("Filtered memory-event rows:", df_events.shape)
display(df_events[["Participant", "V1", "V3", "V4", "V6", "V10"]].head(20))

Raw dataframe shape: (24085, 20)
Columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'Participant']
Filtered memory-event rows: (13440, 20)


,Participant,V1,V3,V4,V6,V10
17,P10,1.613036e+12,[ S3_Memory ] Stimuli Presentation,food_2_e1_11_nr_o_v1_old,nr,NaN
19,P10,1.613036e+12,[ S3_Memory ] Memory Quiz Response,seen,NaN,NaN
22,P10,1.613036e+12,[ S3_Memory ] Stimuli Placed,food_2_e1_11_nr_o_v1_old,nr,Distance : 222.2831
23,P10,1.613036e+12,[ S3_Memory ] Memory Placement confidence,23,NaN,NaN
25,P10,1.613036e+12,[ S3_Memory ] Memory Quiz2 Response,two_weeks,NaN,NaN
26,P10,1.613036e+12,[ S3_Memory ] Memory Quiz2 confidence,37,NaN,NaN
27,P10,1.613036e+12,[ S3_Memory ] Stimuli Presentation,drink_11_memo,ne,NaN
29,P10,1.613036e+12,[ S3_Memory ] Memory Quiz Response,not_seen,NaN,NaN
32,P10,1.613036e+12,[ S3_Memory ] Stimuli Placed,drink_11_memo,ne,Distance : 1118.311
33,P10,1.613036e+12,[ S3_Memory ] Memory Placement confidence,100,NaN,NaN


In [3]:
trials = []
current_trial = {}

for _, row in df_events.iterrows():
    event = str(row["V3"])

    if "Stimuli Presentation" in event:
        if current_trial:
            trials.append(current_trial)

        current_trial = {
            "Participant": row["Participant"],
            "timestamp": row["V1"],
            "stimulus": row["V4"],
            "stimulus_cat_raw": row["V6"],
        }

    elif "Memory Quiz Response" in event:
        current_trial["recognition_response"] = row["V4"]

    elif "Stimuli Placed" in event:
        try:
            dist_str = str(row["V10"])
            distance = float(dist_str.split(":")[1].strip())
        except Exception:
            distance = np.nan

        current_trial["placement_distance"] = distance
        current_trial["placement_raw_V10"] = row["V10"]

    elif "Memory Placement confidence" in event:
        current_trial["placement_confidence"] = row["V4"]

    elif "Memory Quiz2 Response" in event:
        current_trial["temporal_response"] = row["V4"]

    elif "Memory Quiz2 confidence" in event:
        current_trial["temporal_confidence"] = row["V4"]

if current_trial:
    trials.append(current_trial)

mem = pd.DataFrame(trials)

print("Memory trial dataframe shape:", mem.shape)
print("Expected if 35 participants: 35 × 64 = 2240")
display(mem.head(10))

Memory trial dataframe shape: (2240, 10)
Expected if 35 participants: 35 × 64 = 2240


,Participant,timestamp,stimulus,stimulus_cat_raw,recognition_response,placement_distance,placement_raw_V10,placement_confidence,temporal_response,temporal_confidence
0,P10,1.613036e+12,food_2_e1_11_nr_o_v1_old,nr,seen,222.283100,Distance : 222.2831,23,two_weeks,37
1,P10,1.613036e+12,drink_11_memo,ne,not_seen,1118.311000,Distance : 1118.311,100,never,100
2,P10,1.613036e+12,wealth_6_e2_15_r_o_v1_old,re,seen,543.870500,Distance : 543.8705,4,yesterday,46
3,P10,1.613037e+12,pirate_2_e3_11_nr_h_v1_old,nr,not_seen,754.491100,Distance : 754.4911,78,never,78
4,P10,1.613037e+12,viking_8_e3_16_nr_h_v1_old,nr,not_seen,745.093800,Distance : 745.0938,32,never,21
5,P10,1.613037e+12,food_12_memo,ne,not_seen,1268.135000,Distance : 1268.135,100,never,100
6,P10,1.613037e+12,pirate_7_e3_1_nr_h_v1_old,nr,seen,0.952219,Distance : 0.9522193,78,two_weeks,82
7,P10,1.613037e+12,wealth_8_e2_14_r_o_v1_old,re,not_seen,1113.542000,Distance : 1113.542,67,never,74
8,P10,1.613037e+12,pirate_5_e1_7_nr_h_v1_old,nr,seen,348.247200,Distance : 348.2472,18,one_week,35
9,P10,1.613037e+12,viking_7_e1_13_r_h_v1_old,re,not_seen,1189.011000,Distance : 1189.011,57,never,64


In [4]:
def parse_stimulus_name(filename):
    name = str(filename).replace(".jpg", "")
    parts = name.split("_")

    # Old encoded stimuli:
    # food_2_e1_11_nr_o_v1_old
    if "old" in parts and len(parts) >= 8:
        return pd.Series({
            "stim_cat": parts[0],
            "list_no": parts[1],
            "encoding_session": parts[2],
            "checkpoint": parts[3],
            "relevance": parts[4],
            "human_object": parts[5],
            "version": parts[6],
            "old_new": "old",
        })

    # New foils in memory task
    if "memo" in parts or "new" in parts:
        return pd.Series({
            "stim_cat": parts[0] if len(parts) > 0 else np.nan,
            "list_no": np.nan,
            "encoding_session": np.nan,
            "checkpoint": parts[1] if len(parts) > 1 else np.nan,
            "relevance": np.nan,
            "human_object": np.nan,
            "version": np.nan,
            "old_new": "new_memory",
        })

    return pd.Series({
        "stim_cat": parts[0] if len(parts) > 0 else np.nan,
        "list_no": np.nan,
        "encoding_session": np.nan,
        "checkpoint": np.nan,
        "relevance": np.nan,
        "human_object": np.nan,
        "version": np.nan,
        "old_new": np.nan,
    })

parsed = mem["stimulus"].apply(parse_stimulus_name)
mem = pd.concat([mem, parsed], axis=1)

mem["participant_id"] = (
    mem["Participant"]
    .astype(str)
    .str.replace("P", "", regex=False)
    .astype(int)
)

mem["is_old"] = mem["old_new"].eq("old")
mem["is_new_memory"] = mem["old_new"].eq("new_memory")
mem["is_relevant"] = mem["relevance"].map({"r": True, "nr": False})
mem["is_human"] = mem["human_object"].map({"h": True, "o": False})

for col in ["placement_distance", "placement_confidence", "temporal_confidence"]:
    mem[col] = pd.to_numeric(mem[col], errors="coerce")

print("Parsed old/new counts:")
print(mem["old_new"].value_counts(dropna=False))

print("\nVersion counts for old stimuli:")
print(mem[mem["is_old"]]["version"].value_counts(dropna=False))

display(mem.head(10))

Parsed old/new counts:
old_new
old           1680
new_memory     560
Name: count, dtype: int64

Version counts for old stimuli:
version
v1    480
v4    432
v2    384
v3    384
Name: count, dtype: int64


,Participant,timestamp,stimulus,stimulus_cat_raw,recognition_response,placement_distance,placement_raw_V10,placement_confidence,temporal_response,temporal_confidence,...,checkpoint,relevance,human_object,version,old_new,participant_id,is_old,is_new_memory,is_relevant,is_human
0,P10,1.613036e+12,food_2_e1_11_nr_o_v1_old,nr,seen,222.283100,Distance : 222.2831,23,two_weeks,37,...,11,nr,o,v1,old,10,True,False,False,False
1,P10,1.613036e+12,drink_11_memo,ne,not_seen,1118.311000,Distance : 1118.311,100,never,100,...,11,NaN,NaN,NaN,new_memory,10,False,True,NaN,NaN
2,P10,1.613036e+12,wealth_6_e2_15_r_o_v1_old,re,seen,543.870500,Distance : 543.8705,4,yesterday,46,...,15,r,o,v1,old,10,True,False,True,False
3,P10,1.613037e+12,pirate_2_e3_11_nr_h_v1_old,nr,not_seen,754.491100,Distance : 754.4911,78,never,78,...,11,nr,h,v1,old,10,True,False,False,True
4,P10,1.613037e+12,viking_8_e3_16_nr_h_v1_old,nr,not_seen,745.093800,Distance : 745.0938,32,never,21,...,16,nr,h,v1,old,10,True,False,False,True
5,P10,1.613037e+12,food_12_memo,ne,not_seen,1268.135000,Distance : 1268.135,100,never,100,...,12,NaN,NaN,NaN,new_memory,10,False,True,NaN,NaN
6,P10,1.613037e+12,pirate_7_e3_1_nr_h_v1_old,nr,seen,0.952219,Distance : 0.9522193,78,two_weeks,82,...,1,nr,h,v1,old,10,True,False,False,True
7,P10,1.613037e+12,wealth_8_e2_14_r_o_v1_old,re,not_seen,1113.542000,Distance : 1113.542,67,never,74,...,14,r,o,v1,old,10,True,False,True,False
8,P10,1.613037e+12,pirate_5_e1_7_nr_h_v1_old,nr,seen,348.247200,Distance : 348.2472,18,one_week,35,...,7,nr,h,v1,old,10,True,False,False,True
9,P10,1.613037e+12,viking_7_e1_13_r_h_v1_old,re,not_seen,1189.011000,Distance : 1189.011,57,never,64,...,13,r,h,v1,old,10,True,False,True,True


In [5]:
fmri_subject_dirs = sorted([
    p for p in Path("data/fmriprep_output").glob("sub-P*")
    if p.is_dir()
])

fmri_participants = sorted(set([
    int(re.search(r"sub-P(\d+)", p.name).group(1))
    for p in fmri_subject_dirs
    if re.fullmatch(r"sub-P\d+", p.name)
]))

print("fMRI participants:", fmri_participants)
print("N fMRI participants:", len(fmri_participants))

mem_fmri = mem[mem["participant_id"].isin(fmri_participants)].copy()

print("\nBehavioral participants before filter:", mem["participant_id"].nunique())
print("Participants after fMRI filter:", mem_fmri["participant_id"].nunique())

print("\nRows per participant after fMRI filter:")
print(mem_fmri.groupby("participant_id").size().describe())

print("\nOld/new counts after fMRI filter:")
print(mem_fmri["old_new"].value_counts(dropna=False))

print("\nUnparsed rows after fMRI filter:")
display(mem_fmri[mem_fmri["old_new"].isna()][
    ["Participant", "participant_id", "stimulus", "stimulus_cat_raw"]
])

fMRI participants: [10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27, 30, 31, 34, 35, 36, 37, 41, 42, 43, 44, 45, 47, 48, 49, 50, 51]
N fMRI participants: 32

Behavioral participants before filter: 35
Participants after fMRI filter: 32

Rows per participant after fMRI filter:
count    32.0
mean     64.0
std       0.0
min      64.0
25%      64.0
50%      64.0
75%      64.0
max      64.0
dtype: float64

Old/new counts after fMRI filter:
old_new
old           1536
new_memory     512
Name: count, dtype: int64

Unparsed rows after fMRI filter:


,Participant,participant_id,stimulus,stimulus_cat_raw


In [6]:
mem_fmri["recognition_correct"] = np.where(
    mem_fmri["is_old"],
    mem_fmri["recognition_response"].eq("seen"),
    mem_fmri["recognition_response"].eq("not_seen")
)

week_to_temporal_response = {
    "e1": "two_weeks",
    "e2": "one_week",
    "e3": "yesterday",
}

mem_fmri["expected_temporal_response"] = mem_fmri["encoding_session"].map(
    week_to_temporal_response
)
mem_fmri.loc[mem_fmri["is_new_memory"], "expected_temporal_response"] = "never"

mem_fmri["true_temporal_correct"] = mem_fmri["temporal_response"].eq(
    mem_fmri["expected_temporal_response"]
)

print("Recognition accuracy by old/new:")
print(mem_fmri.groupby("old_new")["recognition_correct"].mean())

print("\nTemporal accuracy by old/new:")
print(mem_fmri.groupby("old_new")["true_temporal_correct"].mean())

print("\nOld-item temporal accuracy by encoding session:")
print(mem_fmri[mem_fmri["is_old"]].groupby("encoding_session")["true_temporal_correct"].mean())

Recognition accuracy by old/new:
old_new
new_memory    0.953125
old           0.858724
Name: recognition_correct, dtype: float64

Temporal accuracy by old/new:
old_new
new_memory    0.953125
old           0.387370
Name: true_temporal_correct, dtype: float64

Old-item temporal accuracy by encoding session:
encoding_session
e1    0.425781
e2    0.386719
e3    0.349609
Name: true_temporal_correct, dtype: float64


In [7]:
old = mem_fmri[mem_fmri["is_old"]].copy()

print("Participant × version table:")
display(pd.crosstab(old["participant_id"], old["version"]))

category_week_mapping = (
    old[old["relevance"] == "r"]
    .groupby(["version", "stim_cat", "encoding_session"])
    .size()
    .reset_index(name="n")
    .sort_values(["version", "encoding_session", "stim_cat"])
)

print("Relevant category-week mapping:")
display(category_week_mapping)

mapping_matrix = (
    category_week_mapping
    .pivot_table(
        index=["version", "stim_cat"],
        columns="encoding_session",
        values="n",
        fill_value=0
    )
)

print("Schema matrix by version:")
display(mapping_matrix)

Participant × version table:


version,v1,v2,v3,v4
participant_id,,,,
10,48,0,0,0
12,0,48,0,0
13,0,48,0,0
14,0,0,48,0
15,0,0,48,0
16,0,0,0,48
17,0,0,0,48
18,48,0,0,0
19,48,0,0,0


Relevant category-week mapping:


,version,stim_cat,encoding_session,n
0,v1,drink,e1,28
4,v1,viking,e1,28
3,v1,pirate,e2,28
5,v1,wealth,e2,28
1,v1,food,e3,28
2,v1,maya,e3,28
9,v2,pirate,e1,32
11,v2,wealth,e1,32
7,v2,food,e2,32
8,v2,maya,e2,32


Schema matrix by version:


encoding_session    e1    e2    e3
version stim_cat                  
v1      drink     28.0   0.0   0.0
        food       0.0   0.0  28.0
        maya       0.0   0.0  28.0
        pirate     0.0  28.0   0.0
        viking    28.0   0.0   0.0
        wealth     0.0  28.0   0.0
v2      drink      0.0   0.0  32.0
        food       0.0  32.0   0.0
        maya       0.0  32.0   0.0
        pirate    32.0   0.0   0.0
        viking     0.0   0.0  32.0
        wealth    32.0   0.0   0.0
v3      drink      0.0  32.0   0.0
        food      32.0   0.0   0.0
        maya      32.0   0.0   0.0
        pirate     0.0   0.0  32.0
        viking     0.0  32.0   0.0
        wealth     0.0   0.0  32.0
v4      drink      0.0  36.0   0.0
        food      36.0   0.0   0.0
        maya      36.0   0.0   0.0
        pirate     0.0   0.0  36.0
        viking     0.0  36.0   0.0
        wealth     0.0   0.0  36.0

In [8]:
category_to_schema_week = (
    old[old["relevance"] == "r"]
    .groupby(["version", "stim_cat"])["encoding_session"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

old["schema_predicted_encoding_session"] = old.apply(
    lambda row: category_to_schema_week.get((row["version"], row["stim_cat"])),
    axis=1
)

old["schema_predicted_temporal_response"] = old["schema_predicted_encoding_session"].map(
    week_to_temporal_response
)

old["schema_congruent"] = old["schema_predicted_encoding_session"].eq(
    old["encoding_session"]
)

old["followed_schema_response"] = old["temporal_response"].eq(
    old["schema_predicted_temporal_response"]
)

print("Relevance × schema congruence:")
display(pd.crosstab(old["is_relevant"], old["schema_congruent"], margins=True))

print("\nNormalized:")
display(pd.crosstab(old["is_relevant"], old["schema_congruent"], normalize="index"))

display(old[[
    "participant_id",
    "stimulus",
    "version",
    "stim_cat",
    "encoding_session",
    "relevance",
    "schema_predicted_encoding_session",
    "schema_congruent",
    "temporal_response",
    "expected_temporal_response",
    "schema_predicted_temporal_response",
    "true_temporal_correct",
    "followed_schema_response",
]].head(15))

Relevance × schema congruence:


schema_congruent,False,True,All
is_relevant,,,
False,768,0,768
True,0,768,768
All,768,768,1536



Normalized:


schema_congruent,False,True
is_relevant,,
False,1.0,0.0
True,0.0,1.0


,participant_id,stimulus,version,stim_cat,encoding_session,relevance,schema_predicted_encoding_session,schema_congruent,temporal_response,expected_temporal_response,schema_predicted_temporal_response,true_temporal_correct,followed_schema_response
0,10,food_2_e1_11_nr_o_v1_old,v1,food,e1,nr,e3,False,two_weeks,two_weeks,yesterday,True,False
2,10,wealth_6_e2_15_r_o_v1_old,v1,wealth,e2,r,e2,True,yesterday,one_week,one_week,False,False
3,10,pirate_2_e3_11_nr_h_v1_old,v1,pirate,e3,nr,e2,False,never,yesterday,one_week,False,False
4,10,viking_8_e3_16_nr_h_v1_old,v1,viking,e3,nr,e1,False,never,yesterday,two_weeks,False,False
6,10,pirate_7_e3_1_nr_h_v1_old,v1,pirate,e3,nr,e2,False,two_weeks,yesterday,one_week,False,False
7,10,wealth_8_e2_14_r_o_v1_old,v1,wealth,e2,r,e2,True,never,one_week,one_week,False,False
8,10,pirate_5_e1_7_nr_h_v1_old,v1,pirate,e1,nr,e2,False,one_week,two_weeks,one_week,False,True
9,10,viking_7_e1_13_r_h_v1_old,v1,viking,e1,r,e1,True,never,two_weeks,two_weeks,False,False
10,10,wealth_7_e1_12_nr_o_v1_old,v1,wealth,e1,nr,e2,False,yesterday,two_weeks,one_week,False,False
11,10,maya_1_e3_14_r_h_v1_old,v1,maya,e3,r,e3,True,never,yesterday,yesterday,False,False


In [9]:
irrelevant_old = old[old["is_relevant"] == False].copy()

irrelevant_old["memory_label"] = np.select(
    [
        irrelevant_old["recognition_response"].eq("not_seen"),
        irrelevant_old["true_temporal_correct"],
        irrelevant_old["followed_schema_response"],
    ],
    [
        "forgotten",
        "true_week",
        "schema_week",
    ],
    default="other_wrong_week"
)

print("Irrelevant old response labels:")
print(irrelevant_old["memory_label"].value_counts())
print("\nProportions:")
print(irrelevant_old["memory_label"].value_counts(normalize=True))

counts_by_subject = pd.crosstab(
    irrelevant_old["participant_id"],
    irrelevant_old["memory_label"]
)

display(counts_by_subject)
display(counts_by_subject.describe())

Irrelevant old response labels:
memory_label
schema_week         332
true_week           183
other_wrong_week    131
forgotten           122
Name: count, dtype: int64

Proportions:
memory_label
schema_week         0.432292
true_week           0.238281
other_wrong_week    0.170573
forgotten           0.158854
Name: proportion, dtype: float64


memory_label,forgotten,other_wrong_week,schema_week,true_week
participant_id,,,,
10,12,4,5,3
12,12,1,3,8
13,3,3,14,4
14,0,8,8,8
15,0,9,5,10
16,0,1,19,4
17,6,6,7,5
18,2,0,16,6
19,5,6,10,3


memory_label,forgotten,other_wrong_week,schema_week,true_week
count,32.000000,32.000000,32.000000,32.000000
mean,3.812500,4.093750,10.375000,5.718750
std,3.805238,2.607178,4.419531,2.726624
min,0.000000,0.000000,3.000000,1.000000
25%,0.750000,2.000000,7.000000,4.000000
50%,2.000000,4.000000,10.500000,5.500000
75%,6.250000,6.000000,13.250000,7.250000
max,12.000000,9.000000,19.000000,12.000000


In [10]:
# Create fMRI-ready label table for old irrelevant images
irrelevant_labels = irrelevant_old.copy()

irrelevant_labels["sub"] = irrelevant_labels["participant_id"].apply(lambda x: f"sub-P{x}")

# Viewing event filenames include ".jpg", memory task filenames often do not
irrelevant_labels["event_filename"] = irrelevant_labels["stimulus"].astype(str).apply(
    lambda x: x if x.endswith(".jpg") else x + ".jpg"
)

irrelevant_labels["recognized_seen"] = irrelevant_labels["recognition_response"].eq("seen")

cols = [
    "sub",
    "participant_id",
    "stimulus",
    "event_filename",
    "version",
    "stim_cat",
    "encoding_session",
    "checkpoint",
    "relevance",
    "is_relevant",
    "recognition_response",
    "recognized_seen",
    "temporal_response",
    "expected_temporal_response",
    "schema_predicted_temporal_response",
    "true_temporal_correct",
    "followed_schema_response",
    "memory_label",
    "placement_distance",
    "placement_confidence",
    "temporal_confidence",
]

irrelevant_labels = irrelevant_labels[cols].copy()

print(irrelevant_labels.shape)
print(irrelevant_labels["memory_label"].value_counts())
display(irrelevant_labels.head())

# Save
Path("data/preprocessed").mkdir(parents=True, exist_ok=True)
irrelevant_labels.to_csv(
    "data/preprocessed/irrelevant_old_memory_labels.csv",
    index=False
)

print("Saved: data/preprocessed/irrelevant_old_memory_labels.csv")

(768, 21)
memory_label
schema_week         332
true_week           183
other_wrong_week    131
forgotten           122
Name: count, dtype: int64


,sub,participant_id,stimulus,event_filename,version,stim_cat,encoding_session,checkpoint,relevance,is_relevant,...,recognized_seen,temporal_response,expected_temporal_response,schema_predicted_temporal_response,true_temporal_correct,followed_schema_response,memory_label,placement_distance,placement_confidence,temporal_confidence
0,sub-P10,10,food_2_e1_11_nr_o_v1_old,food_2_e1_11_nr_o_v1_old.jpg,v1,food,e1,11,nr,False,...,True,two_weeks,two_weeks,yesterday,True,False,true_week,222.283100,23,37
3,sub-P10,10,pirate_2_e3_11_nr_h_v1_old,pirate_2_e3_11_nr_h_v1_old.jpg,v1,pirate,e3,11,nr,False,...,False,never,yesterday,one_week,False,False,forgotten,754.491100,78,78
4,sub-P10,10,viking_8_e3_16_nr_h_v1_old,viking_8_e3_16_nr_h_v1_old.jpg,v1,viking,e3,16,nr,False,...,False,never,yesterday,two_weeks,False,False,forgotten,745.093800,32,21
6,sub-P10,10,pirate_7_e3_1_nr_h_v1_old,pirate_7_e3_1_nr_h_v1_old.jpg,v1,pirate,e3,1,nr,False,...,True,two_weeks,yesterday,one_week,False,False,other_wrong_week,0.952219,78,82
8,sub-P10,10,pirate_5_e1_7_nr_h_v1_old,pirate_5_e1_7_nr_h_v1_old.jpg,v1,pirate,e1,7,nr,False,...,True,one_week,two_weeks,one_week,False,True,schema_week,348.247200,18,35


Saved: data/preprocessed/irrelevant_old_memory_labels.csv


In [11]:
# Check whether behavioral labels match viewing-task event filenames

check_rows = []

for sub, sub_df in irrelevant_labels.groupby("sub"):
    pre_event_file = Path("data/bids_ackbar") / sub / "ses-01" / "func" / f"{sub}_ses-01_task-viewing_run-01_events.tsv"
    post_event_file = Path("data/bids_ackbar") / sub / "ses-05" / "func" / f"{sub}_ses-05_task-viewing_run-01_events.tsv"
    
    if not pre_event_file.exists() or not post_event_file.exists():
        check_rows.append({
            "sub": sub,
            "pre_exists": pre_event_file.exists(),
            "post_exists": post_event_file.exists(),
            "n_labels": len(sub_df),
            "n_missing_pre": np.nan,
            "n_missing_post": np.nan,
        })
        continue
    
    pre_events = pd.read_csv(pre_event_file, sep="\t")
    post_events = pd.read_csv(post_event_file, sep="\t")
    
    pre_names = set(pre_events["event"].astype(str))
    post_names = set(post_events["event"].astype(str))
    label_names = set(sub_df["event_filename"].astype(str))
    
    check_rows.append({
        "sub": sub,
        "pre_exists": True,
        "post_exists": True,
        "n_labels": len(label_names),
        "n_missing_pre": len(label_names - pre_names),
        "n_missing_post": len(label_names - post_names),
        "n_pre_matches": len(label_names & pre_names),
        "n_post_matches": len(label_names & post_names),
    })

match_check = pd.DataFrame(check_rows)

display(match_check)
print("\nSummary:")
display(match_check.describe(include="all"))

,sub,pre_exists,post_exists,n_labels,n_missing_pre,n_missing_post,n_pre_matches,n_post_matches
0,sub-P10,True,True,24,0.0,0.0,24.0,24.0
1,sub-P12,True,True,24,0.0,0.0,24.0,24.0
2,sub-P13,True,True,24,0.0,0.0,24.0,24.0
3,sub-P14,True,True,24,0.0,0.0,24.0,24.0
4,sub-P15,True,True,24,0.0,0.0,24.0,24.0
5,sub-P16,True,True,24,4.0,4.0,20.0,20.0
6,sub-P17,True,True,24,4.0,4.0,20.0,20.0
7,sub-P18,True,True,24,0.0,0.0,24.0,24.0
8,sub-P19,True,True,24,0.0,0.0,24.0,24.0
9,sub-P20,True,True,24,0.0,0.0,24.0,24.0



Summary:


,sub,pre_exists,post_exists,n_labels,n_missing_pre,n_missing_post,n_pre_matches,n_post_matches
count,32,32,32,32.0,31.000000,31.000000,31.000000,31.000000
unique,32,1,2,NaN,NaN,NaN,NaN,NaN
top,sub-P10,True,True,NaN,NaN,NaN,NaN,NaN
freq,1,32,31,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,24.0,1.161290,1.032258,22.838710,22.967742
std,NaN,NaN,NaN,0.0,1.845658,1.779211,1.845658,1.779211
min,NaN,NaN,NaN,24.0,0.000000,0.000000,20.000000,20.000000
25%,NaN,NaN,NaN,24.0,0.000000,0.000000,20.000000,22.000000
50%,NaN,NaN,NaN,24.0,0.000000,0.000000,24.000000,24.000000
75%,NaN,NaN,NaN,24.0,4.000000,2.000000,24.000000,24.000000


In [12]:
# Recognized irrelevant old images only
irrelevant_seen_labels = irrelevant_labels[
    irrelevant_labels["recognized_seen"]
].copy()

print("Recognized irrelevant old labels:", irrelevant_seen_labels.shape)
print("\nMemory label counts:")
print(irrelevant_seen_labels["memory_label"].value_counts())
print("\nProportions:")
print(irrelevant_seen_labels["memory_label"].value_counts(normalize=True))

# Subject-level trial counts
recognized_counts_by_subject = pd.crosstab(
    irrelevant_seen_labels["participant_id"],
    irrelevant_seen_labels["memory_label"]
)

display(recognized_counts_by_subject)
display(recognized_counts_by_subject.describe())

# Subjects feasible for true_week vs schema_week contrast
valid_subjects_seen = recognized_counts_by_subject[
    (recognized_counts_by_subject.get("true_week", 0) >= 3) &
    (recognized_counts_by_subject.get("schema_week", 0) >= 3)
].index.tolist()

print("Valid subjects for recognized-only true_week vs schema_week contrast:")
print(valid_subjects_seen)
print("N valid subjects:", len(valid_subjects_seen))

# Save label file
irrelevant_seen_labels.to_csv(
    "data/preprocessed/irrelevant_seen_old_memory_labels.csv",
    index=False
)

print("Saved: data/preprocessed/irrelevant_seen_old_memory_labels.csv")

Recognized irrelevant old labels: (646, 21)

Memory label counts:
memory_label
schema_week         332
true_week           183
other_wrong_week    131
Name: count, dtype: int64

Proportions:
memory_label
schema_week         0.513932
true_week           0.283282
other_wrong_week    0.202786
Name: proportion, dtype: float64


memory_label,other_wrong_week,schema_week,true_week
participant_id,,,
10,4,5,3
12,1,3,8
13,3,14,4
14,8,8,8
15,9,5,10
16,1,19,4
17,6,7,5
18,0,16,6
19,6,10,3


memory_label,other_wrong_week,schema_week,true_week
count,32.000000,32.000000,32.000000
mean,4.093750,10.375000,5.718750
std,2.607178,4.419531,2.726624
min,0.000000,3.000000,1.000000
25%,2.000000,7.000000,4.000000
50%,4.000000,10.500000,5.500000
75%,6.000000,13.250000,7.250000
max,9.000000,19.000000,12.000000


Valid subjects for recognized-only true_week vs schema_week contrast:
[10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 25, 26, 27, 31, 34, 35, 36, 37, 41, 42, 43, 44, 45, 47, 49, 50, 51]
N valid subjects: 29
Saved: data/preprocessed/irrelevant_seen_old_memory_labels.csv


In [13]:
valid_subjects_seen = set(valid_subjects_seen)

irrelevant_seen_valid = irrelevant_seen_labels[
    irrelevant_seen_labels["participant_id"].isin(valid_subjects_seen)
].copy()

print("Valid recognized irrelevant labels:", irrelevant_seen_valid.shape)
print("Subjects:", irrelevant_seen_valid["participant_id"].nunique())
print("\nMemory label counts:")
print(irrelevant_seen_valid["memory_label"].value_counts())

print("\nSubject-level counts:")
display(pd.crosstab(
    irrelevant_seen_valid["participant_id"],
    irrelevant_seen_valid["memory_label"]
))

irrelevant_seen_valid.to_csv(
    "data/preprocessed/irrelevant_seen_old_memory_labels_valid_subjects.csv",
    index=False
)

print("Saved: data/preprocessed/irrelevant_seen_old_memory_labels_valid_subjects.csv")

Valid recognized irrelevant labels: (593, 21)
Subjects: 29

Memory label counts:
memory_label
schema_week         296
true_week           178
other_wrong_week    119
Name: count, dtype: int64

Subject-level counts:


memory_label,other_wrong_week,schema_week,true_week
participant_id,,,
10,4,5,3
12,1,3,8
13,3,14,4
14,8,8,8
15,9,5,10
16,1,19,4
17,6,7,5
18,0,16,6
19,6,10,3


Saved: data/preprocessed/irrelevant_seen_old_memory_labels_valid_subjects.csv


In [15]:
check_rows = []

for sub, sub_df in irrelevant_seen_valid.groupby("sub"):
    pre_event_file = Path("data/bids_ackbar") / sub / "ses-01" / "func" / f"{sub}_ses-01_task-viewing_run-01_events.tsv"
    post_event_file = Path("data/bids_ackbar") / sub / "ses-05" / "func" / f"{sub}_ses-05_task-viewing_run-01_events.tsv"
    
    row = {
        "sub": sub,
        "n_labels": sub_df["event_filename"].nunique(),
        "pre_exists": pre_event_file.exists(),
        "post_exists": post_event_file.exists(),
        "n_missing_pre": np.nan,
        "n_missing_post": np.nan,
        "n_pre_matches": np.nan,
        "n_post_matches": np.nan,
    }
    
    if pre_event_file.exists():
        pre_events = pd.read_csv(pre_event_file, sep="\t")
        pre_names = set(pre_events["event"].astype(str))
        label_names = set(sub_df["event_filename"].astype(str))
        row["n_missing_pre"] = len(label_names - pre_names)
        row["n_pre_matches"] = len(label_names & pre_names)
        
    if post_event_file.exists():
        post_events = pd.read_csv(post_event_file, sep="\t")
        post_names = set(post_events["event"].astype(str))
        label_names = set(sub_df["event_filename"].astype(str))
        row["n_missing_post"] = len(label_names - post_names)
        row["n_post_matches"] = len(label_names & post_names)
    
    check_rows.append(row)

match_check_seen_valid = pd.DataFrame(check_rows)

display(match_check_seen_valid)
display(match_check_seen_valid.describe(include="all"))

,sub,n_labels,pre_exists,post_exists,n_missing_pre,n_missing_post,n_pre_matches,n_post_matches
0,sub-P10,12,True,True,0,0.0,12,12.0
1,sub-P12,12,True,True,0,0.0,12,12.0
2,sub-P13,21,True,True,0,0.0,21,21.0
3,sub-P14,24,True,True,0,0.0,24,24.0
4,sub-P15,24,True,True,0,0.0,24,24.0
5,sub-P16,24,True,True,4,4.0,20,20.0
6,sub-P17,18,True,True,3,3.0,15,15.0
7,sub-P18,22,True,True,0,0.0,22,22.0
8,sub-P19,19,True,True,0,0.0,19,19.0
9,sub-P20,23,True,True,0,0.0,23,23.0


,sub,n_labels,pre_exists,post_exists,n_missing_pre,n_missing_post,n_pre_matches,n_post_matches
count,29,29.000000,29,29,29.000000,28.000000,29.000000,28.000000
unique,29,NaN,1,2,NaN,NaN,NaN,NaN
top,sub-P10,NaN,True,True,NaN,NaN,NaN,NaN
freq,1,NaN,29,28,NaN,NaN,NaN,NaN
mean,NaN,20.448276,NaN,NaN,1.034483,0.928571,19.413793,19.392857
std,NaN,3.896759,NaN,NaN,1.721352,1.653920,4.523883,4.441715
min,NaN,12.000000,NaN,NaN,0.000000,0.000000,10.000000,10.000000
25%,NaN,18.000000,NaN,NaN,0.000000,0.000000,15.000000,16.500000
50%,NaN,22.000000,NaN,NaN,0.000000,0.000000,21.000000,20.500000
75%,NaN,24.000000,NaN,NaN,3.000000,0.750000,23.000000,23.000000


In [16]:
clean_label_subjects = match_check_seen_valid[
    (match_check_seen_valid["pre_exists"]) &
    (match_check_seen_valid["post_exists"]) &
    (match_check_seen_valid["n_missing_pre"].eq(0)) &
    (match_check_seen_valid["n_missing_post"].eq(0))
]["sub"].tolist()

print("Clean subjects with full pre/post event matching:")
print(clean_label_subjects)
print("N:", len(clean_label_subjects))

Clean subjects with full pre/post event matching:
['sub-P10', 'sub-P12', 'sub-P13', 'sub-P14', 'sub-P15', 'sub-P18', 'sub-P19', 'sub-P20', 'sub-P21', 'sub-P23', 'sub-P27', 'sub-P31', 'sub-P34', 'sub-P35', 'sub-P41', 'sub-P44', 'sub-P45', 'sub-P47', 'sub-P49', 'sub-P50']
N: 20


In [17]:
irrelevant_seen_clean = irrelevant_seen_valid[
    irrelevant_seen_valid["sub"].isin(clean_label_subjects)
].copy()

print("Clean labels:", irrelevant_seen_clean.shape)
print("Subjects:", irrelevant_seen_clean["sub"].nunique())
print(irrelevant_seen_clean["memory_label"].value_counts())

irrelevant_seen_clean.to_csv(
    "data/preprocessed/irrelevant_seen_old_memory_labels_clean_fmri.csv",
    index=False
)

print("Saved: data/preprocessed/irrelevant_seen_old_memory_labels_clean_fmri.csv")

Clean labels: (416, 21)
Subjects: 20
memory_label
schema_week         203
true_week           127
other_wrong_week     86
Name: count, dtype: int64
Saved: data/preprocessed/irrelevant_seen_old_memory_labels_clean_fmri.csv


In [18]:
clean_counts = pd.crosstab(
    irrelevant_seen_clean["sub"],
    irrelevant_seen_clean["memory_label"]
)

display(clean_counts)
display(clean_counts.describe())

memory_label,other_wrong_week,schema_week,true_week
sub,,,
sub-P10,4,5,3
sub-P12,1,3,8
sub-P13,3,14,4
sub-P14,8,8,8
sub-P15,9,5,10
sub-P18,0,16,6
sub-P19,6,10,3
sub-P20,6,10,7
sub-P21,5,6,6


memory_label,other_wrong_week,schema_week,true_week
count,20.000000,20.000000,20.000000
mean,4.300000,10.150000,6.350000
std,2.696977,4.648429,2.739093
min,0.000000,3.000000,3.000000
25%,3.000000,6.750000,4.000000
50%,4.000000,10.000000,6.500000
75%,6.000000,14.000000,8.000000
max,9.000000,19.000000,12.000000


In [19]:
sub = "sub-P10"

labels = irrelevant_seen_clean[irrelevant_seen_clean["sub"] == sub].copy()

event_file = Path("data/bids_ackbar") / sub / "ses-01" / "func" / f"{sub}_ses-01_task-viewing_run-01_events.tsv"
events_pre = pd.read_csv(event_file, sep="\t")

label_map = dict(zip(labels["event_filename"], labels["memory_label"]))

events_pre_labeled = events_pre.copy()

events_pre_labeled["memory_label"] = events_pre_labeled["event"].map(label_map)

events_pre_labeled["trial_type_memory"] = events_pre_labeled["memory_label"].map({
    "true_week": "irrelevant_seen_true_week",
    "schema_week": "irrelevant_seen_schema_week",
    "other_wrong_week": "irrelevant_seen_other_wrong_week",
})

events_pre_labeled["trial_type_memory"] = events_pre_labeled["trial_type_memory"].fillna("other")

print("Original events:", events_pre.shape)
print("Labels for subject:", labels.shape)
print("\nMemory label counts in labels:")
print(labels["memory_label"].value_counts())

print("\nTrial type counts in viewing events:")
print(events_pre_labeled["trial_type_memory"].value_counts())

display(events_pre_labeled[[
    "onset", "duration", "event", "memory_label", "trial_type_memory"
]].head(30))

Original events: (384, 7)
Labels for subject: (12, 21)

Memory label counts in labels:
memory_label
schema_week         5
other_wrong_week    4
true_week           3
Name: count, dtype: int64

Trial type counts in viewing events:
trial_type_memory
other                               312
irrelevant_seen_schema_week          30
irrelevant_seen_other_wrong_week     24
irrelevant_seen_true_week            18
Name: count, dtype: int64


,onset,duration,event,memory_label,trial_type_memory
0,4.634,3.017,maya_3_e3_6_r_h_v1_old.jpg,NaN,other
1,11.701,3.017,wealth_5_e3_7_nr_o_v1_old.jpg,other_wrong_week,irrelevant_seen_other_wrong_week
2,17.934,3.017,pirate_6_e2_13_r_h_v1_old.jpg,NaN,other
3,24.534,3.017,viking_11_new.jpg,NaN,other
4,30.451,3.016,pirate_7_e3_1_nr_h_v1_old.jpg,other_wrong_week,irrelevant_seen_other_wrong_week
5,37.151,3.016,food_8_e3_13_r_o_v1_old.jpg,NaN,other
6,43.367,3.017,pirate_8_e2_6_r_h_v1_old.jpg,NaN,other
7,48.551,3.016,food_9_new.jpg,NaN,other
8,54.367,3.017,viking_3_e2_1_nr_h_v1_old.jpg,schema_week,irrelevant_seen_schema_week
9,61.800,3.017,food_6_e2_2_nr_o_v1_old.jpg,true_week,irrelevant_seen_true_week


In [20]:
out_dir = Path("data/preprocessed/relabeled_events")
out_dir.mkdir(parents=True, exist_ok=True)

label_file = Path("data/preprocessed/irrelevant_seen_old_memory_labels_clean_fmri.csv")
labels_all = pd.read_csv(label_file)

label_to_trial_type = {
    "true_week": "irrelevant_seen_true_week",
    "schema_week": "irrelevant_seen_schema_week",
    "other_wrong_week": "irrelevant_seen_other_wrong_week",
}

summary_rows = []

for sub, sub_labels in labels_all.groupby("sub"):
    label_map = dict(zip(sub_labels["event_filename"], sub_labels["memory_label"]))
    
    for ses in ["ses-01", "ses-05"]:
        event_file = (
            Path("data/bids_ackbar") / sub / ses / "func" /
            f"{sub}_{ses}_task-viewing_run-01_events.tsv"
        )
        
        if not event_file.exists():
            print(f"Missing event file: {event_file}")
            continue
        
        events = pd.read_csv(event_file, sep="\t")
        events_labeled = events.copy()
        
        events_labeled["memory_label"] = events_labeled["event"].map(label_map)
        events_labeled["trial_type_memory"] = events_labeled["memory_label"].map(label_to_trial_type)
        events_labeled["trial_type_memory"] = events_labeled["trial_type_memory"].fillna("other")
        
        save_file = out_dir / f"{sub}_{ses}_task-viewing_run-01_events_memory_labels.tsv"
        events_labeled.to_csv(save_file, sep="\t", index=False)
        
        counts = events_labeled["trial_type_memory"].value_counts().to_dict()
        summary_rows.append({
            "sub": sub,
            "ses": ses,
            "save_file": str(save_file),
            **counts,
        })

summary_relabeled_events = pd.DataFrame(summary_rows).fillna(0)

display(summary_relabeled_events)
print("Saved relabeled event files to:", out_dir)

,sub,ses,save_file,other,irrelevant_seen_schema_week,irrelevant_seen_other_wrong_week,irrelevant_seen_true_week
0,sub-P10,ses-01,data/preprocessed/relabeled_events/sub-P10_ses...,312,30,24.0,18
1,sub-P10,ses-05,data/preprocessed/relabeled_events/sub-P10_ses...,312,30,24.0,18
2,sub-P12,ses-01,data/preprocessed/relabeled_events/sub-P12_ses...,312,18,6.0,48
3,sub-P12,ses-05,data/preprocessed/relabeled_events/sub-P12_ses...,312,18,6.0,48
4,sub-P13,ses-01,data/preprocessed/relabeled_events/sub-P13_ses...,258,84,18.0,24
5,sub-P13,ses-05,data/preprocessed/relabeled_events/sub-P13_ses...,258,84,18.0,24
6,sub-P14,ses-01,data/preprocessed/relabeled_events/sub-P14_ses...,240,48,48.0,48
7,sub-P14,ses-05,data/preprocessed/relabeled_events/sub-P14_ses...,240,48,48.0,48
8,sub-P15,ses-01,data/preprocessed/relabeled_events/sub-P15_ses...,240,30,54.0,60
9,sub-P15,ses-05,data/preprocessed/relabeled_events/sub-P15_ses...,240,30,54.0,60


Saved relabeled event files to: data/preprocessed/relabeled_events
